In [5]:
import torch
from typing import Optional
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class RewardHead(nn.Module):
    """
    RewardHead类给GPT2实现了一个“头”，为每个输出的token返回一个标量值。
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.reward = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(self.reward.weight, std=(1.0 / np.sqrt(self.hidden_size + 1)))
        nn.init.zeros_(self.reward.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.reward(output)


class GPT2RewardModel(nn.Module):
    """
    GPT2模型加上一个“奖励头”
    """

    def __init__(self, model_name):
        super().__init__()
        self.llm = AutoModelForCausalLM.from_pretrained(model_name)
        # 添加奖励头
        self.reward_head = RewardHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:
        # GPT2的输出
        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )

        # 获取最后一层隐藏层
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # 对隐藏层给出奖励
        rewards = self.reward_head(last_hidden_state).squeeze(-1)
        # 归一化
        return torch.sigmoid(rewards)

In [6]:
model_name = "./gpt2"
reward_model = GPT2RewardModel(model_name)
reward_model.load_state_dict(torch.load("reward_model.pt", map_location='cpu'))

<All keys matched successfully>

In [7]:
import torch
from typing import Optional
from torch import nn
import numpy as np
from transformers import AutoModelForCausalLM

class ValueHead(nn.Module):
    """
    ValueHead类为GPT2实现了一个“头”，会为输出的每个token返回一个标量值
    标量值就是这个token的价值，ValueHead就是评论家。
    """

    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.value = nn.Linear(self.hidden_size, 1)
        self._post_init()

    def _post_init(self):
        nn.init.normal_(self.value.weight, std=(1.0 / np.sqrt(self.hidden_size + 1)))
        nn.init.zeros_(self.value.bias)

    def forward(self, hidden_states):
        output = hidden_states
        return self.value(output)

In [8]:
class ModelForCausalLMWithValueHead(nn.Module):
    """
    GPT2模型+一个价值头
    """

    def __init__(self, model_path):
        super().__init__()
        # 这个要初始化为我们微调出来的gpt2-sft模型
        # actor演员模型
        self.llm = AutoModelForCausalLM.from_pretrained(model_path)
        # 添加价值头
        # critic评论家模型
        self.v_head = ValueHead(self.llm.config)

    def forward(
        self,
        input_ids,
        attention_mask,
    ) -> Optional[torch.FloatTensor]:
        # gpt2-sft模型的输出
        transformer_outputs = self.llm.forward(
            input_ids,
            attention_mask=attention_mask,
            output_hidden_states = True,
        )
        # 输出的token
        lm_logits = transformer_outputs.logits
        # 获取最后一层隐藏层
        last_hidden_state = transformer_outputs.hidden_states[-1]

        # 评估token的价值
        value = self.v_head(last_hidden_state).squeeze(-1)
        # 返回输出的token和token的价值
        return lm_logits, value

    def generate(self, *args, **kwargs):
        return self.llm.generate(*args, **kwargs)

In [9]:
model_path = './gpt2-sft'
model = ModelForCausalLMWithValueHead(model_path)

In [10]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

from datasets import load_dataset
dataset = load_dataset("./sst2")
print(dataset)

ds_train, ds_val = dataset['train'], dataset['validation']

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})


In [11]:
print(len(ds_train))
ds_train = ds_train.filter(lambda x: len(x['sentence'].split(' ')) > 8)
ds_val = ds_val.filter(lambda x: len(x['sentence'].split(' ')) > 8)

print(len(ds_train))
print(len(ds_val))

67349
31105
807


In [12]:
import random
input_min_token_length = 2
input_max_token_length = 8
input_token_length_range = list(range(input_min_token_length, input_max_token_length))
print(input_token_length_range)
print(random.choice(input_token_length_range))

[2, 3, 4, 5, 6, 7]
6


In [13]:
def tokenize(sample):
    input_size = random.choice(input_token_length_range)
    sample['input_ids'] = tokenizer.encode(sample['sentence'])[:input_size]
    sample['attention_mask'] = [1] * len(sample['input_ids'])
    sample['query'] = tokenizer.decode(sample['input_ids'])
    return sample

map_kwargs = {
    "batched": False,
    "remove_columns": ['idx', 'sentence', 'label']
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

In [14]:
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')

print(tokenized_dataset_train[6])

{'input_ids': tensor([1640,  883, 3807]), 'attention_mask': tensor([1, 1, 1]), 'query': 'for those movie'}


In [15]:
REWARD_TOKEN_ID = tokenizer.eos_token_id

In [16]:
from torch.utils.data import DataLoader

batch_size = 32

def collator(batch):
    return dict((key, [d[key] for d in batch]) for key in batch[0])

train_dataloader = DataLoader(tokenized_dataset_train, batch_size=batch_size, collate_fn=collator, shuffle=True)
val_dataloader = DataLoader(tokenized_dataset_val, batch_size=batch_size, collate_fn=collator, shuffle=True)

batch = next(iter(train_dataloader))
print(batch)

{'input_ids': [tensor([ 272,  625,  301, 2645, 1143,  837, 1308]), tensor([ 1350, 13504,   262,  2126,   286,  1521]), tensor([18108,   645,  5876]), tensor([ 11, 644, 705,  82]), tensor([ 4480,   663,  2426,  2300,   287,   257, 14854]), tensor([   64, 12625,  4065]), tensor([5661,  318]), tensor([  732,   705,   260, 12908,   510,   287]), tensor([11545,  5895,   326,   285,    13]), tensor([18820,  1327,   284,   307,  8258,   287]), tensor([1101, 1654,  612,  705,   82,  257]), tensor([  338,   355, 22066]), tensor([   7,  479, 1370]), tensor([  11, 4437,  705,   82]), tensor([ 4491,   933,   869,   283, 19623, 31432,   710]), tensor([  805,  1095,  1239,   284,  1663, 14262]), tensor([ 1078,  1791,   284,  2222, 44182]), tensor([   11, 14187, 22874,   837]), tensor([ 1659,   262,  1266, 34549]), tensor([13116,   326,   340]), tensor([ 8873,   560, 10721]), tensor([ 1169,  5884,  3923,   286,  1449, 17868,   290]), tensor([13959,  1613,  2636,   318,   588]), tensor([23108,   286, 

In [17]:
output_min_length = 5
output_max_length = 16

# https://huggingface.co/docs/trl/how_to_train#how-to-generate-text-for-training
# gpt2-sft输出的配置
generation_kwargs = {
    "min_length": -1,
    "top_k": 0.0,
    "top_p": 1.0,
    "do_sample": True,
    "pad_token_id": tokenizer.pad_token_id
}

In [18]:
new_tokens = random.choice(list(range(output_min_length, output_max_length)))
generation_kwargs["max_new_tokens"] = new_tokens
sample = tokenizer('Hi, this')
print(sample)

{'input_ids': [17250, 11, 428], 'attention_mask': [1, 1, 1]}


In [19]:
query_response = model.generate(
    input_ids=torch.tensor(sample['input_ids']).unsqueeze(0),
    attention_mask=torch.tensor(sample['attention_mask']).unsqueeze(0),
    **generation_kwargs
).squeeze(0)
print(query_response)

tensor([17250,    11,   428,  9439,   318,  1016,   284,   307])


In [20]:
print(tokenizer.decode(query_response))

Hi, this tomorrow is going to be


In [21]:
with torch.no_grad():
    query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID])])
    attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
    score = reward_model(
        query_response_score.unsqueeze(0),
        attention_mask.unsqueeze(0)
    ).squeeze(0)[-1]
print(score)

tensor(0.8867)


In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
reward_model = reward_model.to(device)

query_tensors = batch['input_ids']
query_attention_masks = batch['attention_mask']

response_tensors = []
query_response_tensors = []
score_tensors = []

for i, query in enumerate(query_tensors):
    query = query.to(device)
    query_attention_mask = query_attention_masks[i].to(device)
    new_tokens = random.choice(list(range(output_min_length, output_max_length)))
    generation_kwargs["max_new_tokens"] = new_tokens
    query_response = model.generate(
        input_ids=query.unsqueeze(0),
        attention_mask=query_attention_mask.unsqueeze(0),
        **generation_kwargs
    ).squeeze(0)

    response_len = len(query_response) - len(query)
    response_tensors.append(query_response[-response_len:])
    query_response_tensors.append(query_response)

    with torch.no_grad():
        query_response_score = torch.cat([query_response, torch.tensor([REWARD_TOKEN_ID]).to(device)])
        attention_mask = torch.ones_like(query_response_score, dtype=torch.long)
        score = reward_model(
            query_response_score.unsqueeze(0),
            attention_mask.unsqueeze(0)
        ).squeeze(0)[-1]
        score = 2 * (score - 0.5)
    score_tensors.append(score)

batch["response"] = [tokenizer.decode(response) for response in response_tensors]
from pprint import pprint
pprint(batch['response'])

['dy somewhat comically twisted version of violence that should',
 ' do bad films change movies more frequently when',
 ' believing that pure greek astrology',
 ' missing from this movie is the genuine excitement .',
 'eful and delicately crafted fashion . ',
 ' comedy that is icky',
 ' a surprisingly fair game for fans of the spy and at least',
 ' this sweet , dark , very romantic comedy iced by ace romero',
 ' m. macdonald lives a well-',
 ' this genre         ',
 ' good chance somebody else might be playing',
 ' as you can get      ',
 ' ) lacks formal knowledge and is too earnest',
 ' assignations to perspicacious documentary work      would have',
 ' williams as a',
 ' .             ',
 ' between the acclaimed and beloved cast ',
 ' jordan and mattel executives make great front seats , but this film',
 ' cast ia ia ia ia ia .',
 ' was modeled on real-life events ?"',
 " , raunchy violence , and iani 's happy ending",
 ' arnold ershowitz ',
 ' mediocre entertainment .          ',
 

In [24]:
from copy import deepcopy
sft_model = deepcopy(model)

In [25]:
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

input_data = data_collator([
    {'input_ids': ids,
     'attention_mask': torch.ones_like(ids)} for ids in query_response_tensors
]).to(device)
print(input_data)

{'input_ids': tensor([[  272,   625,   301,  2645,  1143,   837,  1308,  9892,  6454,   401,
          1146, 19074,  2196,   286,  3685,   326,   815, 50256, 50256, 50256,
         50256],
        [ 1350, 13504,   262,  2126,   286,  1521,   466,  2089,  7328,  1487,
          6918,   517,  6777,   618, 50256, 50256, 50256, 50256, 50256, 50256,
         50256],
        [18108,   645,  5876, 14773,   326,  5899,   308, 10316,  6468, 31142,
         50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256],
        [   11,   644,   705,    82,  4814,   422,   428,  3807,   318,   262,
          8768, 14067,   764, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256],
        [ 4480,   663,  2426,  2300,   287,   257, 14854, 13839,   290,  8675,
          1286, 18025,  6977,   764,   220, 50256, 50256, 50256, 50256, 50256,
         50256],
        [   64, 12625,  4065, 10997,   326,   318,   220, 17479, 50256, 50256,
         50256, 50256, 50256, 50

In [83]:
def compute_rewards(input_data, query_tensors, response_tensors, score_tensors):
    with torch.no_grad():
        # 正在微调的模型所输出的token的logits和token的价值
        # 模型输出所有token的概率分布
        logits, values = model(**input_data) # b, seq, vocab
        # 冻结的模型的输出和价值
        ref_logits, _ = sft_model(**input_data)
        # 正在微调的模型的输出的对数概率
        logp = torch.nn.functional.log_softmax(logits[:, :-1, :], dim=-1)
        # 冻结的模型的输出的对数概率
        ref_logp = torch.nn.functional.log_softmax(ref_logits[:, :-1, :], dim=-1)
        # 实际生成的token序列
        labels = input_data['input_ids'][:, 1:] # b, seq
        # 使用gather提取实际token的概率
        logp = torch.gather(logp, 2, labels.unsqueeze(-1)).squeeze(-1) # batch, seq
        ref_logp = torch.gather(ref_logp, 2, labels.unsqueeze(-1)).squeeze(-1) # batch, seq
        # kl散度
        kl = logp - ref_logp
        # kl散度的权重
        beta = 0.2
        # 最终奖励的计算
        rewards = - beta * kl
        attention_mask = input_data['attention_mask']
        masks = torch.zeros_like(attention_mask[:, 1:])
        masks[:,:] = attention_mask[:, 1:]
        flag = False
        for j in range(len(query_tensors)):
            start = len(query_tensors[j]) - 1
            end = start + len(response_tensors[j])
            masks[j, :start] = 0
            masks[j, end:] = 0
            print(rewards[j])
            rewards[j, end - 1] += score_tensors[j]
            print(rewards[j])
            rewards[j, :] *= masks[j, :]
            values[j, :-1] *= masks[j, :]
            if not flag:
                print(tokenizer.decode(input_data['input_ids'][j]))
                print(tokenizer.decode(input_data['input_ids'][j][:-1]))
                print(tokenizer.decode(input_data['input_ids'][j][1:]))
                print(tokenizer.decode(input_data['input_ids'][j][start:end+1]))
                print(tokenizer.decode(input_data['input_ids'][j][1:][end-1]))
                print('start: ', start)
                print('end: ', end)
                flag = True

    return logp, rewards, values[:, :-1], masks

In [84]:
logprobs, rewards, values, masks = compute_rewards(input_data, query_tensors, response_tensors, score_tensors)

tensor([-0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.],
       device='cuda:0')
tensor([-0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000,
        -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.0000, -0.9143,
        -0.0000, -0.0000, -0.0000, -0.0000], device='cuda:0')
an overstylized , purdy somewhat comically twisted version of violence that should<|endoftext|><|endoftext|><|endoftext|><|endoftext|>
an overstylized , purdy somewhat comically twisted version of violence that should<|endoftext|><|endoftext|><|endoftext|>
 overstylized , purdy somewhat comically twisted version of violence that should<|endoftext|><|endoftext|><|endoftext|><|endoftext|>
 purdy somewhat comically twisted version of violence that should
 should
start:  6
end:  16
tensor([-0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0., -0.],
       device='cuda:0')
tensor([-0.0000, -0.0000, -0.

In [ ]:
def masked_mean(values, mask):
    return (values * mask).sum() / mask.sum()

def masked_var(values, mask):
    mean = masked_mean(values, mask)
    centred_values = values - mean
    return masked_mean(centred_values ** 2, mask)

def masked_whiten(values, mask):
    mean, var = masked_mean(values, mask), masked_var(values, mask)
    whitened = (values - mean) * torch.rsqrt(var + 1e-8)
    whitened += mean
    return whitened

def compute_advantage(rewards, values, masks):
    lastgae = 0.0
    advantage_reversed = []
    seq_length = rewards.shape[-1]
    gamma, lam = 1.0, 0.95

    for t in reversed(range(seq_length)):
        nextvalues = values[:, t + 1] if t < seq_length - 1 else 0.0
        delta = rewards[:, t] + gamma * nextvalues - values[:, t]
        lastgae = delta + gamma * lam * lastgae
        advantage_reversed.append(lastgae)
    advantages = torch.stack(advantage_reversed[::-1], dim=1)
    # 归一化一下
    advantages = masked_whiten(advantages, masks)

    returns = advantages + values
    return advantages, returns

In [ ]:
advantages, returns = compute_advantage(rewards, values, masks)
print(advantages[0])
print(returns[0])

In [ ]:
learning_rate = 1e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
# 重新排列一下各个批次
np.random.permutation(batch_size)

In [ ]:
mini_batch_size = 4
ppo_epochs = 4

cliprange_ratio = 0.2

v_loss_coeff = 0.1

ratio_threshold = 10

def compute_loss(
    old_logprobs,
    values,
    logprobs,
    vpreds,
    masks,
    advantages,
    returns
):
    ratio = torch.exp(logprobs - old_logprobs)
    pg_loss1 = - ratio * advantages
    pg_loss2 = - torch.clamp(
        ratio,
        1 - cliprange_ratio,
        1 + cliprange_ratio
    ) * advantages
    pg_loss = masked_mean(torch.max(pg_loss1, pg_loss2), masks)

    v_loss = masked_mean((vpreds - returns) ** 2, masks)
    loss = pg_loss + v_loss_coeff * v_loss

    avg_ratio = masked_mean(ratio, masks)
    if avg_ratio > ratio_threshold:
        pg_loss = pg_loss * 0.0
        v_loss = v_loss * 0.0
        loss = loss * 0.0

    return loss, v_loss

def mini_batch_train():
    # 过滤掉输入数据为空的批次
    if input_data['input_ids'].shape[0] == 0:
        return
    for ep in range(ppo_epochs):
        batch_inds = np.random.permutation(batch_size)

        for start in range(0, batch_size, mini_batch_size):
            end = start + mini_batch_size
            mini_batch_inds = batch_inds[start:end]

            mb_model_inputs = {
                'input_ids': input_data['input_ids'][mini_batch_inds],
                'attention_mask': input_data['attention_mask'][mini_batch_inds]
            }
            mb_logits, mb_vpreds = model(**mb_model_inputs)
            mb_logits = torch.nn.functional.log_softmax(
                mb_logits[:, :-1, :],
                dim=-1
            )
            mb_logprobs = torch.gather(
                mb_logits,
                2,
                mb_model_inputs['input_ids'][:, 1:].unsqueeze(-1)
            ).squeeze(-1)

            loss, loss_v = compute_loss(
                logprobs[mini_batch_inds],
                values[mini_batch_inds],
                mb_logprobs,
                mb_vpreds[:, :-1],
                masks[mini_batch_inds],
                advantages[mini_batch_inds],
                returns[mini_batch_inds]
            )

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            print('loss/total', loss.item())
    print('mini-batch training finished')

In [ ]:
mini_batch_train()

In [ ]:
num_epochs = 1

for epoch in range(num_epochs):
    for batch in train_dataloader:
        # Generate responses
        query_tensors = batch['input_ids']
        query_attention_masks = batch['attention_mask']

        response_tensors = []
        query_response_tensors = []
        score_tensors = []

        for i, query in enumerate(query_tensors):
            query = query.to(device)
            query_attention_mask = query_attention_masks[i].to(device)
            new_tokens = random.choice(list(range(
                output_min_length,
                output_max_length)))
            generation_kwargs["max_new_tokens"] = new_tokens
            query_response = model.generate(
                input_ids=query.unsqueeze(0),
                attention_mask=query_attention_mask.unsqueeze(0),
                **generation_kwargs
            ).squeeze(0)

            response_len = len(query_response) - len(query)
            response_tensors.append(query_response[-response_len:])
            query_response_tensors.append(query_response)

            with torch.no_grad():
                query_response_score = torch.cat([
                    query_response,
                    torch.tensor([REWARD_TOKEN_ID]).to(device)])
                attention_mask = torch.ones_like(
                    query_response_score,
                    dtype=torch.long)
                score = reward_model(
                    query_response_score.unsqueeze(0),
                    attention_mask.unsqueeze(0)
                ).squeeze(0)[-1]
                score = 2 * (score - 0.5)
            score_tensors.append(score)

        input_data = data_collator([
            {
                'input_ids': ids,
                'attention_mask': torch.ones_like(ids)
            }
            for ids in query_response_tensors
        ]).to(device)

        # 奖励和优势
        logprobs, rewards, values, masks = compute_rewards(
            input_data,
            query_tensors,
            response_tensors,
            score_tensors
        )
        advantages, returns = compute_advantage(rewards, values, masks)

        # 小批次训练
        mini_batch_train()
    print(f'epoch {epoch + 1} finished')

In [ ]:
print(len(tokenized_dataset_val))
val_gen_lengths = [0] * len(tokenized_dataset_val)
for i in range(len(tokenized_dataset_val)):
    val_gen_lengths[i] = random.choice(list(range(
        output_min_length,
        output_max_length)))
val_gen_lengths[:10]

In [ ]:
def validate():
    scores = []
    for b, batch in enumerate(val_dataloader):
        # Generate_responses
        query_tensors = batch['input_ids']
        query_attention_masks = batch['attention_mask']
        for i, query in enumerate(query_tensors):
            query = query.to(device)
            query_attention_mask = query_attention_masks[i].to(device)
            new_tokens = val_gen_lengths[b * len(query_tensors) + i]
            generation_kwargs["max_new_tokens"] = new_tokens
            query_response = model.generate(
                input_ids=query.unsqueeze(0),
                attention_mask=query_attention_mask.unsqueeze(0),
                **generation_kwargs
            ).squeeze(0)
            query_response_score = torch.cat([
                query_response,
                torch.tensor([REWARD_TOKEN_ID]).to(device)])
            attention_mask = torch.ones_like(
                query_response_score, dtype=torch.long)
            score = reward_model(
                query_response_score.unsqueeze(0),
                attention_mask.unsqueeze(0)
            ).squeeze(0)[-1]
            score = 2 * (score - 0.5)
            scores.append(score.item())
    print('平均分数:', sum(scores) / len(scores))

In [ ]:
validate()

In [ ]:
torch.save(model.state_dict(), 'gpt2-ppo.pt')

In [ ]:
model_path = './gpt2-sft'
model = ModelForCausalLMWithValueHead(model_path).to(device)
validate()

In [ ]:
from transformers import pipeline, set_seed
from pprint import pprint
g = pipeline('text-generation', model='./gpt2-ppo-without-vhead')
set_seed(1337)
pprint(g("Hi, this is all terribly", max_length=30, num_return_sequences=1))

In [1]:
def quick_model_comparison(model1, model2, threshold=1e-6):
    """快速模型比较"""
    
    print("⚡ 快速模型比较")
    print("="*40)
    
    params1 = dict(model1.named_parameters())
    params2 = dict(model2.named_parameters())
    common_params = set(params1.keys()) & set(params2.keys())
    
    identical_count = 0
    different_count = 0
    significant_diff_count = 0
    
    for name in common_params:
        p1, p2 = params1[name], params2[name]
        
        if p1.shape != p2.shape:
            continue
            
        diff = torch.max(torch.abs(p1 - p2)).item()
        
        if diff < 1e-10:
            identical_count += 1
        elif diff < threshold:
            different_count += 1
        else:
            significant_diff_count += 1
            if significant_diff_count <= 5:  # 只显示前5个
                print(f"🔴 显著差异: {name} (最大差异: {diff:.8f})")
    
    print(f"\n📊 结果统计:")
    print(f"   完全相同的层: {identical_count}")
    print(f"   微小差异的层: {different_count}")
    print(f"   显著差异的层: {significant_diff_count}")
    
    if identical_count == len(common_params):
        print("✅ 两个模型参数完全相同！")
    elif significant_diff_count == 0:
        print("✅ 两个模型参数基本相同（仅有数值精度差异）")
    else:
        print("⚠️ 两个模型存在显著参数差异")
    
    return {
        'identical': identical_count,
        'minor_diff': different_count,
        'significant_diff': significant_diff_count,
        'total_common': len(common_params)
    }

# 使用示例
# result = quick_model_comparison(model, sft_model)


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
model1 = AutoModelForCausalLM.from_pretrained('gpt2-sft')
model2 = AutoModelForCausalLM.from_pretrained('gpt2-ppo-without-vhead')
quick_model_comparison(model1, model2)

⚡ 快速模型比较
🔴 显著差异: transformer.h.3.attn.c_proj.bias (最大差异: 0.00513126)
🔴 显著差异: transformer.h.5.mlp.c_fc.weight (最大差异: 0.01644957)
🔴 显著差异: transformer.h.8.mlp.c_fc.bias (最大差异: 0.00733607)
🔴 显著差异: transformer.h.1.attn.c_attn.bias (最大差异: 0.00756520)
🔴 显著差异: transformer.h.9.mlp.c_proj.weight (最大差异: 0.01394147)

📊 结果统计:
   完全相同的层: 0
   微小差异的层: 0
   显著差异的层: 148
⚠️ 两个模型存在显著参数差异


{'identical': 0, 'minor_diff': 0, 'significant_diff': 148, 'total_common': 148}